# Actual ML Workflow

```text
1. Load Dataset
        ↓
2. Understand the Dataset
        ↓
3. Data Preprocessing
        ↓
4. Exploratory Data Analysis (EDA)
        ↓
5. Feature Engineering (if needed)
        ↓
6. Train-Test Split
        ↓
7. Model Selection
        ↓
8. Train Model
        ↓
9. Evaluate Model
        ↓
10. Improve Model (Hyperparameter Tuning)
```

# 1: Load the Dataset ✅

```python
df = pd.read_csv("train.csv")
```

Nothing else.

# 2: Understand the Dataset (Always)

This is the first thing you should do.

```python
df.head()
df.shape
df.info()
df.describe()
df.columns
df.dtypes
```

You're trying to answer questions like:

- How many rows?
- How many columns?
- Target column?
- Numeric or categorical?
- Missing values?
- Data types?

For your dataset:

- 1460 rows ✅
- 81 columns ✅
- Target = `SalePrice` ✅

# 3: Data Preprocessing (Always)

Now clean the data before training.

## Step 1: Check Missing Values ✅

Run:

```python
df.isnull().sum().sort_values(ascending=False)
```

You'll see something like:

| Column | Missing Values |
|---------|----------------|
| PoolQC | 1453 |
| MiscFeature | 1406 |
| Alley | 1369 |
| Fence | 1179 |
| FireplaceQu | 690 |
| LotFrontage | 259 |
| GarageType | 81 |
| GarageYrBlt | 81 |
| GarageFinish | 81 |
| BsmtQual | 37 |
| ... | ... |

### Why?

Machine learning models cannot handle `NaN` values (most models in scikit-learn). You must deal with them before training.

### Solution:
# Case 1: Almost All Values Are Missing → Drop the Column

Example:

```text
PoolQC    1453 missing
```

Percentage missing:

```text
1453 / 1460 ≈ 99.5%
```

Only 7 values are available.

Keeping this column won't help the model much.

✅ Usually:

```python
df.drop(columns=["PoolQC"], inplace=True)
```
```text
In pandas, inplace=True is used to modify the original DataFrame directly without creating a new copy of the data.
```
---

# Case 2: Some Values Are Missing → Fill (Impute)

Example:

```text
LotFrontage    259 missing
```

Missing percentage:

```text
259 / 1460 ≈ 17.7%
```

We don't want to lose the entire column.

Instead:

```python
df["LotFrontage"].fillna(df["LotFrontage"].median(), inplace=True)
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())
```

### Note:
# A Very Important Point

Don't drop or fill just because there are missing values.

First ask:

**Why are they missing?**

For example:

- **PoolQC (Pool Quality):** If a house has no swimming pool, then `NaN` doesn't mean the data is missing—it means the house doesn't have a pool.
- **Alley:** If a house has no alley access, `NaN` means no alley.
- **Fence:** If a house has no fence, `NaN` means no fence.

In these cases, replacing `NaN` with `"None"` is often more meaningful than treating it as an unknown value.

So, before deciding how to handle missing values, always understand what the column represents.


---

## Step 2: Check Duplicate Rows ✅

```python
df.duplicated().sum()
```

### Why?

Duplicate houses can bias the model and make evaluation less reliable.

---

## Step 3: Check Data Types ✅

```python
df.info()
```

You'll see:

- `int64`
- `float64`
- `object`

### Why?

- `int64` → Numeric
- `float64` → Numeric
- `object` → Categorical (strings)

Most ML models cannot directly use string values, so these will need encoding later.

---

## Step 4: Check Target Column

```python
df["SalePrice"].isnull().sum()
```

It should return:

```python
0
```

### Why?

If the target has missing values, supervised learning becomes difficult because the model needs the correct answer during training.

---

## Step 5: Check Constant Columns

```python
df.nunique().sort_values()
```

### Why?

If a column has only one unique value, it provides no useful information and can be removed.

Example:

| Utilities |
|-----------|
| AllPub |
| AllPub |
| AllPub |
| AllPub |

If every row has the same value, it doesn't help predict house prices.

---

## Step 6: Check Incorrect Data Types

Example:

`MSSubClass`

Although it's stored as an integer, it actually represents categories of building types rather than a quantity. Later, you may treat it as categorical.

---

## Step 7: Encode Categorical Columns (Later)

Columns like:

- Street
- MSZoning
- Neighborhood
- SaleCondition

contain text.

```python
X.select_dtypes(include="object").columns
```
Example:

**Street**

- Pave
- Grvl

These must be converted into numbers before training.

## sloution:
# Convert Categorical Columns

Use **One-Hot Encoding**:

```python
X = pd.get_dummies(X, drop_first=True)  or X = pd.get_dummies(X, columns=["MSZoning", "Street"], drop_first=True) both are work same but first one is better
```

Now every column becomes numeric.

## Example

### Before

| MSZoning |
|-----------|
| RL |
| RM |
| FV |

### After

| MSZoning_RL | MSZoning_RM |
|--------------|-------------|
| 1 | 0 |
| 0 | 1 |
| 0 | 0 |

---

## Step 8: Feature Scaling (Depends on Model)

Not every model needs scaling.

| Model | Scaling Needed? |
|--------|-----------------|
| Linear Regression | ✅ Recommended |
| Logistic Regression | ✅ |
| KNN | ✅ |
| SVM | ✅ |
| Neural Networks | ✅ |
| Decision Tree | ❌ |
| Random Forest | ❌ |
| XGBoost | ❌ |

---

## So, what preprocessing does this dataset need?

| Task | Required? | Why? |
|------|-----------|------|
| Missing values | ✅ Yes | Many columns contain NaN values |
| Duplicate rows | ✅ Check | Remove if any exist |
| Data types | ✅ Check | Understand numeric vs. categorical features |
| Remove useless columns | ✅ Yes | Columns with almost all missing values or constant values may be dropped |
| Encode categorical variables | ✅ Yes | Models require numeric inputs |
| Feature scaling | ⚠️ Depends | Needed for some algorithms, not all |

: